# MARSAD Analytics (v1.4)

Analyses for social media datasets: CrowdTangle, Meltwater, Meta Content Library, X, Reddit (Communalytic), or any CSV, Excel, JSON or Parquet file with a text column. Built for Arabic and English; each card shows a language badge saying whether it works for any language, languages with spaces between words, the Arabic and English lexicons, or a model's languages.

This notebook runs the same code as the MARSAD dashboard (package `marsad_analytics`, embedded below), so results match between the two.

**How to use**
1. Run cell 1 once.
2. Run cell 2 and pick your file(s) with *Choose Files* (several files can be compared). For big files, tick *MOUNT_DRIVE* instead and type the Drive path in the dashboard.
3. Run cell 3. Your files are listed and selected; check the time zone and weekend settings and press *Load and scan*. The scan shows which cards your file supports and names the missing field for the others.
4. Optional: tick INSTALL_MODEL_LIBS in cell 1, switch to a GPU runtime (Runtime > Change runtime type), then open *Enrich* to add sentiment (DistilBERT multilingual), emotions (XLM-EMO), toxicity (Detoxify and TextDetox), hate speech (DE-LIMIT, Arabic and English) and dialect labels. Each post goes to a model that covers its language; other posts are skipped and counted. The *Sentiment, emotion and toxicity* package then shows the results. The MARSAD platform analyses (propaganda, propaganda techniques, political bias, check-worthiness, claims, fact verification) run once you type your API URL or model id next to them in *Enrich*; the *MARSAD platform analyses* package groups them.
5. Start from *Packages* (Basic, Narrative, Audience, Advanced): each says how many of its analyses will run on your file and which are skipped and why, and a suggested starting point is shown. Or use *Research presets*, *Choose your own insights* (by category), *Concordance (KWIC)* or *Search*. *Run and pin* adds a card to the report; *Check robustness* re-runs it without duplicates, without the most active accounts and on random halves.
6. Click any bar, point, heatmap cell or word in a chart to see the posts behind it (matched words highlighted); *Posts for* does the same for table rows. Hover a column header or a headline number to see how it is calculated.
7. Each card has a download bar: Excel (all tables, with an About and an Explanations sheet), CSV per table, and the chart as PNG, SVG or interactive HTML.
8. TOPIC-1 is a topic analyzer (clusters, silhouette score, run history, 3D map) and WHO-4 a network analyzer (reply, quote, mention, link and hashtag networks, with a GraphML download and a standalone network explorer).
9. To compare files, upload them together in cell 2 (or select several in the Files list): each file becomes a dataset, DATA-6 compares them, and most cards can split by dataset.
10. Export the report as an interactive HTML file (clickable charts and tables, works offline), PDF, LaTeX, CSV or JSON.

Cards are named by the question they answer: DATA, TIME, WHO, WORDS, TOPIC, FRAME, REACT, SPREAD, PAPER, plus a number in reading order (1 = look at this first). Old ids such as A1 or C1 still work.

Only a text column is required. Cards that need a missing field stay greyed out and name the field.

Lexicon-based cards are marked exploratory until the lexicon has been validated (cell 5).


In [ ]:
#@title 1. Install (run once) { display-mode: "form" }
HIGH_QUALITY_PDF_CHARTS = False #@param {type:"boolean"}
import sys, subprocess, base64, io, zipfile, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly>=5", "duckdb", "pyarrow", "arabic-reshaper",
                "python-bidi", "openpyxl", "reportlab", "networkx", "kaleido", "matplotlib", "wordfreq", "anywidget", "regex"], check=False)
INSTALL_MODEL_LIBS = False #@param {type:"boolean"}
if INSTALL_MODEL_LIBS:   # transformers and torch are already in Colab; these add toxicity, topics, FastText and sentence embeddings
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "detoxify", "sentence-transformers", "umap-learn",
                    "bertopic", "gensim", "sentencepiece"], check=False)
PKG_DIR = os.path.abspath("marsad_pkg")
PKG_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG_B64))).extractall(PKG_DIR)
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass
if HIGH_QUALITY_PDF_CHARTS:
    try:
        import kaleido; kaleido.get_chrome_sync()
    except Exception as e:
        print("Chrome for chart export not installed; PDF charts will use matplotlib.", e)
import marsad_analytics
print("marsad_analytics", marsad_analytics.__version__, "ready")


In [ ]:
#@title 2. Upload files { display-mode: "form" }
MOUNT_DRIVE = False #@param {type:"boolean"}
import os
UPLOAD_DIR = "/content/marsad_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)
try:
    from google.colab import files, drive
    if MOUNT_DRIVE:
        drive.mount("/content/drive")   # then type e.g. /content/drive/MyDrive/data/export.csv in the dashboard
        print("Drive mounted at /content/drive")
    else:
        up = files.upload()             # Colab's own file picker: click "Choose Files"
        for name, data in up.items():
            with open(os.path.join(UPLOAD_DIR, name), "wb") as f:
                f.write(data)
        print("Saved", len(up), "file(s) to", UPLOAD_DIR)
except ImportError:
    print("Not in Colab: put files in ./marsad_uploads or type a path in the dashboard.")


In [ ]:
#@title 3. Open the dashboard
from marsad_analytics.ui import Dashboard
dash = Dashboard()
dash.show()


In [ ]:
#@title 4. Scripted use (optional): run cards from code on the data loaded above
s = dash.session            # or: from marsad_analytics import load, Session, Settings; s = Session(load("file.csv"), Settings(analysis_tz="Asia/Qatar"))
r = s.run("FRAME-2", lex_a="hope", lex_b="despair", by="time")
r.figure.show() if r.figure is not None else None
display(r.main_table())
print("\n".join(r.caveats))
# More: s.run("REACT-2", feature="hashtags", outcome="shares"); s.run("TIME-4", events=[{"name": "Event", "date": "2024-01-26"}])
# Corpus tools: s.run("WORDS-3", target="group:page", group="owner_type", reference="rest"); s.kwic("غزة"); s.run("WORDS-5", node="غزة")
# Posts behind a chart point: s.examples(r.figure.data[0].customdata[0][0])
# s.add_lexicon("my_terms", ["term one", "كلمة"]); s.run("FRAME-1", lexicons=["my_terms"])
# Models (GPU): from marsad_analytics import enrich; enrich.run_classifier(s, "toxicity"); s.run("FRAME-9", threshold=0.7)
# Topics: s.run("TOPIC-1", min_cluster_size=20, min_samples=10, epsilon=0.1); s.run("TOPIC-1", use_run=1)
# Network: s.run("WHO-4", kind="reply", search="name", hide_pendants=True, date_from="2024-01-01", toxic_only=True)
# from marsad_analytics.cards import network; network.save_graphml(s, "net.graphml"); network.explorer_html(s, "net.html")


In [ ]:
#@title 5. Validate a lexicon (precision on sampled matches)
LEXICON = "hope" #@param {type:"string"}
N = 200 #@param {type:"integer"}
from marsad_analytics import validation
sheet = validation.sample_matches(dash.session, LEXICON, n=N, path=f"validate_{LEXICON}.xlsx")
print(f"Wrote validate_{LEXICON}.xlsx with {len(sheet)} posts. Two annotators fill the 'correct' column (1 or 0),")
print("then run: res = validation.score(['validate_hope_A.xlsx', 'validate_hope_B.xlsx'], dataset='name', annotators=['A','B'])")
try:
    from google.colab import files; files.download(f"validate_{LEXICON}.xlsx")
except Exception:
    pass


## Card reference

Old indicator ids (F1 to F138 and the primary analyses) are listed so earlier results can be traced.

| Card | Name | Question | Replaces |
| --- | --- | --- | --- |
| DATA-1 | Dataset overview | What does this file contain, and which cards can run on it? | A1 |
| DATA-2 | Text statistics | How large and what shape is the text? | A2, T1, T2, T3, T4, F111 |
| DATA-3 | Languages and dialects | Which scripts and dialects are in the data? | A5 |
| DATA-4 | Duplicates | How much of the dataset is repeated text? | A3, F106 (replacement) |
| DATA-5 | Off-topic check | How much of the collected data is actually on topic? | A4 |
| DATA-6 | Compare files | How do two or more datasets differ? | H1, F120, F121 |
| TIME-1 | Posts over time | How does activity change over time? | B1, F12 (replacement) |
| TIME-2 | Posting hours and days | At what hours and days do people post, and when do posts get more engagement? | B2, F8, F10, F13 |
| TIME-3 | Spikes | Which days show unusual spikes? | B4, F52, F89 |
| TIME-4 | Before and after an event | Did activity or framing change around a known event? | B3, F63 (generic part), F113 (generic part) |
| WHO-1 | Top accounts | Who posts most and who gets most engagement? | C1, F36 |
| WHO-2 | Concentration | Is activity or attention concentrated in a few actors, countries or categories? | C2, F21, F37, F41 |
| WHO-3 | Posting regularity | How regularly do actors post, and how stable is their engagement? | C3, F9, F11, F45 |
| WHO-4 | Network analyzer | Who interacts with whom, and which communities form? | C7 |
| WHO-5 | Unusual accounts | Are there accounts with unusual volume, regularity or repetition? | C8 |
| WHO-6 | Countries | Where do posts come from? | C5, F20, F23 |
| WHO-7 | Page categories | Which kinds of Facebook pages drive the conversation? | C4, F39, F83 |
| WHO-8 | Page age | Do older pages get more engagement? | C6, F38 |
| WORDS-1 | Word list | Which words, phrases, hashtags and mentions are most frequent, and how widely are they spread? | L1, D1, Keyword Analysis |
| WORDS-2 | Word cloud | What does the vocabulary look like at a glance? | L8 |
| WORDS-3 | Keywords | Which words are unusually frequent here compared with a reference? | L2, D2 |
| WORDS-4 | Concordance (KWIC) | In what contexts is a word or phrase used? | L3 |
| WORDS-5 | Collocates | Which words keep company with a given word? | L4 |
| WORDS-6 | Phrases around a word | Which fixed phrases form around a word? | L5 |
| WORDS-7 | Term over time | When, and by how many accounts, is a term used? | L6, D3, F103, F84 (generic part), F117 (generic part) |
| WORDS-8 | Vocabulary variety | Do groups differ in how varied their vocabulary is? | L7 |
| WORDS-9 | Emoji | Which emoji are used, and by whom? | D7 |
| TOPIC-1 | Topics | What are the main topics, and how do they change? | D6 |
| TOPIC-2 | Word similarity map | Which words are used in similar contexts? | D4 |
| TOPIC-3 | Posts close to a concept | Which posts are semantically close to a concept? | D5 |
| FRAME-1 | Lexicon use over time | How often does a kind of language appear, and how does that change? | E1, F54, F58, F64, F68, F70, F73, F74 ... |
| FRAME-2 | Two lexicons compared | Does one framing dominate another, and does that change over time? | E2, F53, F56, F60, F61, F66, F88, F109 |
| FRAME-3 | Lexicons in the same post | Do two or three kinds of language appear in the same posts? | E3, F78, F101, F125, F132 |
| FRAME-4 | Pronouns and certainty | Who is the text about, and how strongly does it commit? | E4, F104, F108 |
| FRAME-5 | Language profile by group | Do groups use different language? | E5, F80, F119 (generic), F122 (generic), F123, F124, F128 |
| FRAME-6 | Your own word list | How often does my own list of terms appear? | E6, classify_solicitation_language |
| FRAME-7 | Sentiment | Is the text positive, negative or neutral (or any other classifier's labels)? | E7 |
| FRAME-8 | Stance toward a target | Do posts support or oppose a given target? | E8 |
| FRAME-9 | Toxicity | How toxic is the language, and how does it change? |  |
| FRAME-10 | Emotions | Which emotions does the text express? |  |
| FRAME-11 | Hate speech | Which posts are hate speech? |  |
| FRAME-12 | Propaganda | Which posts use propaganda? |  |
| FRAME-13 | Propaganda techniques | Which propaganda techniques are used? |  |
| FRAME-14 | Political bias | Which political leaning do the posts show? |  |
| REACT-1 | Engagement overview | How is engagement distributed, and how much do a few posts dominate? | R1, engagement_summary, outlier_detection, virality, correlation_analysis |
| REACT-2 | What gets engagement | Which post features go with more engagement? | R3, F3, F4, F5, F6, F17, F25, F26 ... |
| REACT-3 | Lexicon and engagement | Do posts using a kind of language get more engagement? | R4, F57, F71, F85, F93, F96, F102, F116 ... |
| REACT-4 | Engagement mix | What kind of engagement do posts get? | R5, F1, F7, F33, F72, F75 |
| REACT-5 | Reaction types | Which Facebook reactions do posts get, and how does that change? | R6, F49, F50, F138 |
| REACT-6 | Reaction sentiment | Is the audience response positive, divided or unified? | R7, sentiment_analysis, sentiment_sensitivity_analysis, F24, F31, F51, F86, F87 |
| REACT-7 | Engagement rate | How much engagement relative to audience size? | R2, F32, calculate_engagement_rate |
| REACT-8 | Views | How much engagement per view, and how many views per follower? | G1, F34, F35 |
| REACT-9 | Overperforming posts | Which posts did better than their page usually does? | R8, overperforming_analysis, F19 |
| REACT-10 | Video performance | How do video length, ownership and share status relate to engagement? | R9, F2, F14, F15, F16, F18, video_analysis |
| SPREAD-1 | Repeated claims | Which wordings spread, how fast, and through how many accounts? | G6 |
| SPREAD-2 | Linked websites | Which sources are linked, and how many links are hidden behind shorteners? | G5 |
| SPREAD-3 | Coordinated posting | Do groups of accounts post the same content at the same moment, repeatedly? | G4, F133 (replacement) |
| SPREAD-4 | Reply threads | How deep and how large are reply threads? | G7 |
| SPREAD-5 | Crosspost reach | How far do views travel beyond the original post? | G2, F46, F47, F48, F127 |
| SPREAD-6 | Meltwater reach and sources | What reach, sources and vendor sentiment does Meltwater report? | G3, MeltwaterExtras |
| SPREAD-7 | Check-worthy posts | Which posts make claims worth fact-checking? |  |
| SPREAD-8 | Claims | Which claims are made, and which are repeated most? |  |
| SPREAD-9 | Fact verification | Are the claims supported or refuted? |  |
| SOCIETY-1 | Social cohesion | How connected and united is the conversation? |  |
| SOCIETY-2 | Emotional intensity | How heated is the writing? |  |
| SOCIETY-3 | Distress and resilience | Do people express distress, or resilience and hope? |  |
| SOCIETY-4 | Economic concern | How much do people talk about prices, jobs, poverty and corruption? |  |
| SOCIETY-5 | Calls to action | Do posts call people to act, join or find solutions? |  |
| SOCIETY-6 | Blame and accountability | Who is blamed, and do people ask for accountability? |  |
| SOCIETY-7 | Identity and belonging | How do people talk about who they are? |  |
| SOCIETY-8 | Trust in institutions | Do people trust or distrust government, media, courts, health and science? |  |
| SOCIETY-9 | Moral values | Which moral values do posts appeal to: care, fairness, loyalty, authority, sanctity? |  |
| SOCIETY-10 | Wellbeing | Do posts show positive feelings, engagement, relationships, meaning and accomplishment? |  |
| SOCIETY-11 | Stress and coping | How much stress do people express, and how do they cope? |  |
| SOCIETY-12 | Certainty and self-focus | Do people write with certainty or doubt, and about 'I' or 'we'? |  |
| SOCIETY-13 | Economic mood | Is economic talk about worries and shortages, or good news? |  |
| SOCIETY-14 | Boycotts and buying calls | Do people call to boycott products, or to buy local? |  |
| SOCIETY-15 | Health talk | Do people ask for help, and how do they talk about vaccines? |  |
| SOCIETY-16 | Misinformation signals | How much conspiracy talk and miracle-cure talk is there? |  |
| SOCIETY-17 | Issue attention | Which public issues get the most attention? |  |
| PAPER-1 | Report export | How do I get results into a paper? | I1 |
| PAPER-2 | Annotation sample | Which posts should we annotate, and do annotators agree? | I2 |
| PAPER-3 | Datasheet | What goes in the dataset description of the paper? | I3 |

**Presets**

| Preset | Cards | For |
| --- | --- | --- |
| Quick profile | DATA-1, DATA-2, TIME-1, WHO-1, WORDS-1, WORDS-2 | Any new file |
| Dataset paper descriptives | DATA-1, DATA-2, DATA-4, DATA-5, DATA-3, TIME-1, WHO-2, WHO-6, WORDS-1, WORDS-3, WORDS-8, TOPIC-1, SPREAD-6, PAPER-3 | Corpus statistics sections of dataset papers |
| Annotation round | DATA-3, TOPIC-1, PAPER-2 | Draw a sample, then check agreement |
| Misinformation and spread | SPREAD-2, SPREAD-1, DATA-4, SPREAD-3, WHO-4, WHO-5 | Links, repeated claims, coordination |
| Engagement study | REACT-1, REACT-7, REACT-2, REACT-4, TIME-2, WHO-1 | What drives interactions |
| Discourse and framing study | WORDS-3, WORDS-4, WORDS-5, FRAME-1, FRAME-2, FRAME-3, FRAME-4, FRAME-5, REACT-3, WORDS-7 | Framing, emotion, hope and despair, us versus them |
| Event study | TIME-1, TIME-4, TIME-3, WORDS-7, WORDS-3, FRAME-1 | Before and after a known event |
| Actor and coordination audit | WHO-1, WHO-2, WHO-3, WHO-4, WHO-5, DATA-4, SPREAD-3 | Who dominates, and whether it looks organized |
| Corpus linguistics (AntConc-style) | WORDS-1, WORDS-3, WORDS-4, WORDS-5, WORDS-6, WORDS-7, WORDS-8, WORDS-2 | Word lists, keywords, concordances, collocates |
| Reaction study (CrowdTangle, Meta) | REACT-5, REACT-6, TIME-3 | Facebook reaction data |

**Validation and annotation.** Cell 5 checks a lexicon's precision. For a full annotation round use card PAPER-2 (stratified sample, one sheet per annotator), then `validation.agreement([...sheets])` for Cohen's kappa, Fleiss' kappa and Krippendorff's alpha. Card PAPER-3 drafts the dataset's datasheet.

The Palestine/Gaza topic pack (tick it under *Topic packs* before loading) adds 12 lexicons and 18 preset cards that reproduce the notebook's Gaza-specific indicators.
